# 03b · Piloto StyleGAN2-ADA (corrección de compatibilidad)

**Por qué existe este notebook:** en el piloto 03, StyleGAN2-ADA falló con
`RuntimeError: derivative for aten::grid_sampler_2d_backward is not implemented`.

**Causa:** la regularización R1 necesita derivar dos veces a través del aumento de datos de ADA (`grid_sample`).
El repo trae una versión propia (`grid_sample_gradfix`) que lo resuelve, pero está **desactivada para PyTorch ≥ 1.10** por un control de versión.
En GPU, el `grid_sample` estándar de PyTorch no implementa esa segunda derivada → error.

**Corrección:** se reemplaza `grid_sample_gradfix.py` por la versión del repositorio oficial **StyleGAN3** de NVIDIA (commit `c233a91`), que ya está adaptada a PyTorch ≥ 1.12.
Solo se quita su dependencia de `pkg_resources` (ya no viene con setuptools reciente). Se verificó que da exactamente los mismos gradientes de segundo orden que PyTorch.

**Extensiones CUDA (`bias_act`, `upfirdn2d`):** sí compilaban, pero el repo no lograba cargarlas: en PyTorch moderno `cpp_extension.load()` devuelve el módulo y ya no lo deja importable con `importlib.import_module`. Se usa directamente el módulo que devuelve `load()` (parche 4, probado con una extensión de prueba). Sin esto, el repo usaba la implementación de referencia (correcta, pero más lenta).

**Entorno:** GPU **L4**, versión **2026.07**. Duración ≈ 25 min (la primera compilación de las extensiones tarda 1–2 min). Requiere haber ejecutado el notebook 03 (usa `datasets/train128_stylegan.zip`).

## Paso 1 · Entorno, rutas y dataset

In [2]:
import sys, os, subprocess, json, shutil, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA (torch):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ SIN GPU")

SEED = 42
PILOT_MIN_GAN = float(os.environ.get("PILOT_MIN_GAN", 15))
UNITS_PER_HOUR = 1.8          # ← la tasa real que muestra tu panel de recursos

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())
SG_ZIP = LOCAL_DIR / LOG03["datasets"]["train_zip"]
if not SG_ZIP.exists():
    shutil.copy2(PROJECT_DIR / "datasets" / SG_ZIP.name, SG_ZIP)
assert sha256_file(SG_ZIP) == LOG03["datasets"]["train_sha256"], "⚠️ El dataset no coincide con el del notebook 03"
print("Dataset verificado ✔", SG_ZIP.name)

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | CUDA (torch): 12.8
GPU: NVIDIA L4
Mounted at /content/drive
Dataset verificado ✔ train128_stylegan.zip


## Paso 2 · Clonar el repo oficial y aplicar los 4 parches

In [3]:
%pip install -q click requests tqdm ninja psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.0 MB/s eta 0:00:00


In [4]:
SG_COMMIT = "d72cc7d041b42ec8e806021a205ed9349f87c6a4"      # NVlabs/stylegan2-ada-pytorch
SG3_COMMIT = "c233a919a6faee6e36a316ddd4eddababad1adf9"     # NVlabs/stylegan3 (solo para grid_sample_gradfix.py)
SG_DIR = LOCAL_DIR / "stylegan2-ada-pytorch"
SG3_DIR = LOCAL_DIR / "stylegan3"

def clonar(url, destino, commit):
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", url, str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "checkout", "-q", commit], check=True)
    head = subprocess.run(["git", "-C", str(destino), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == commit, head

clonar("https://github.com/NVlabs/stylegan2-ada-pytorch.git", SG_DIR, SG_COMMIT)
clonar("https://github.com/NVlabs/stylegan3.git", SG3_DIR, SG3_COMMIT)
subprocess.run(["git", "-C", str(SG_DIR), "checkout", "-q", "--", "."], check=True)   # parte de archivos limpios

# Parche 1: distutils (Python >= 3.12)
p = SG_DIR / "dnnlib" / "util.py"; s = p.read_text()
s = s.replace("from distutils.util import strtobool",
              "try:\n    from distutils.util import strtobool\nexcept ImportError:  # Python >= 3.12\n"
              "    def strtobool(v):\n        return v.lower() in ('y', 'yes', 't', 'true', 'on', '1')")
p.write_text(s)

# Parche 2: Sampler.__init__ sin argumentos (PyTorch moderno)
p = SG_DIR / "torch_utils" / "misc.py"; s = p.read_text()
s = s.replace("        super().__init__(dataset)\n", "        super().__init__()\n")
p.write_text(s)

# Parche 3: grid_sample_gradfix de StyleGAN3, sin pkg_resources
s = (SG3_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").read_text()
s = s.replace("from pkg_resources import parse_version\n", "")
s = s.replace("_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
              "_TV = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])\n_use_pytorch_1_11_api = _TV >= (1, 11)")
s = s.replace("_use_pytorch_1_12_api = parse_version(torch.__version__) >= parse_version('1.12.0a') # Allow prerelease builds of 1.12",
              "_use_pytorch_1_12_api = _TV >= (1, 12)")
assert "parse_version" not in s
(SG_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").write_text(s)

# Parche 4: custom_ops — PyTorch moderno compila la extensión y DEVUELVE el módulo con load(),
# pero ya no lo registra para importlib.import_module (por eso fallaba con "No module named ..._plugin")
p = SG_DIR / "torch_utils" / "custom_ops.py"; s = p.read_text()
a = "            torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,"
b = "            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)"
c = "        module = importlib.import_module(module_name)\n"
assert s.count(a) == 1 and s.count(b) == 1 and s.count(c) == 1
s = s.replace(a, "            module = " + a.strip()).replace(b, "            module = " + b.strip()).replace(c, "")
p.write_text(s)

assert "super().__init__(dataset)" not in (SG_DIR / "torch_utils" / "misc.py").read_text()
print("Parches aplicados ✔ (distutils, Sampler, grid_sample_gradfix, custom_ops)")

Parches aplicados ✔ (distutils, Sampler, grid_sample_gradfix, custom_ops)


## Paso 3 · Verificar el parche de `grid_sample` (segunda derivada en GPU)

In [5]:
sys.path.insert(0, str(SG_DIR))
from torch_utils.ops import grid_sample_gradfix as gs
import torch.nn.functional as F
gs.enabled = True
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
x0 = torch.randn(2, 3, 32, 32, device=dev)
g = (torch.rand(2, 32, 32, 2, device=dev) * 2 - 1) * 0.9
x = x0.clone().requires_grad_(True)
y = gs.grid_sample(x, g)
gx, = torch.autograd.grad((y * torch.sin(y)).sum(), x, create_graph=True)
gx.square().sum().backward()                       # aquí fallaba antes en GPU
ref = F.grid_sample(x0, g, mode="bilinear", padding_mode="zeros", align_corners=False)
print("Segunda derivada OK ✔ | diferencia forward vs PyTorch:", float((y - ref).abs().max()))

Segunda derivada OK ✔ | diferencia forward vs PyTorch: 0.0


/tmp/ipykernel_264/1748601312.py:14: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Segunda derivada OK ✔ | diferencia forward vs PyTorch:", float((y - ref).abs().max()))


## Paso 4 · Diagnóstico de las extensiones CUDA (`bias_act`, `upfirdn2d`)
Se intenta compilarlas con salida completa. Si fallan, se muestra el error; el entrenamiento igual funciona con la versión de referencia (más lenta).
`TORCH_CUDA_ARCH_LIST=8.9` corresponde a la arquitectura de la L4 y acelera la compilación.

In [6]:
print(subprocess.run("nvcc --version | tail -2; which nvcc; ninja --version", shell=True,
                     capture_output=True, text=True).stdout)
os.environ["TORCH_EXTENSIONS_DIR"] = str(LOCAL_DIR / "torch_extensions")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"
import traceback
from torch_utils import custom_ops
custom_ops.verbosity = "full"
from torch_utils.ops import bias_act, upfirdn2d
COMPILACION = {}
for nombre, mod in [("bias_act", bias_act), ("upfirdn2d", upfirdn2d)]:
    t0 = time.time()
    try:
        ok = bool(mod._init())
    except Exception:
        ok = False; traceback.print_exc()
    COMPILACION[nombre] = {"compilado": ok, "segundos": round(time.time() - t0, 1)}
print(COMPILACION)

Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
/usr/local/cuda/bin/nvcc
1.13.2.git.kitware.jobserver-pipe-1

Setting up PyTorch plugin "bias_act_plugin"...
Done setting up PyTorch plugin "bias_act_plugin".
Setting up PyTorch plugin "upfirdn2d_plugin"...
Done setting up PyTorch plugin "upfirdn2d_plugin".
{'bias_act': {'compilado': True, 'segundos': 38.0}, 'upfirdn2d': {'compilado': True, 'segundos': 22.7}}


## Paso 5 · Entrenamiento piloto (15 min)
Mismas opciones que el notebook 03: `--cond=1 --mirror=0 --metrics=none --cfg=auto`. Las variables de entorno del Paso 4 se heredan, así que reutiliza las extensiones ya compiladas.

In [7]:
GAN_RUNS = LOCAL_DIR / "sg_piloto_runs"
if GAN_RUNS.exists(): shutil.rmtree(GAN_RUNS)
GAN_LOG = LOGS / "03b_piloto_stylegan_v2.log"
cmd = [sys.executable, "train.py", f"--outdir={GAN_RUNS}", f"--data={SG_ZIP}", "--gpus=1",
       "--cond=1", "--mirror=0", "--metrics=none", "--cfg=auto", "--kimg=1000", "--snap=1000", f"--seed={SEED}"]
print(" ".join(cmd))
t0 = time.time()
with open(GAN_LOG, "w") as flog:
    proc = subprocess.Popen(cmd, cwd=SG_DIR, stdout=flog, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    try:
        proc.wait(timeout=PILOT_MIN_GAN * 60)
    except subprocess.TimeoutExpired:
        proc.terminate()
        try: proc.wait(timeout=30)
        except subprocess.TimeoutExpired: proc.kill()
log_txt = GAN_LOG.read_text(errors="ignore")
print(f"Ejecutó {(time.time()-t0)/60:.1f} min | código de salida: {proc.returncode} (None/-15 = detenido por tiempo, es lo esperado)")
print("\n".join(log_txt.splitlines()[-15:]))

/usr/bin/python3 train.py --outdir=/content/data/sg_piloto_runs --data=/content/data/train128_stylegan.zip --gpus=1 --cond=1 --mirror=0 --metrics=none --cfg=auto --kimg=1000 --snap=1000 --seed=42
Ejecutó 15.0 min | código de salida: -15 (None/-15 = detenido por tiempo, es lo esperado)
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
/content/data/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.11.0+cu128. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
/content/data/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.11.0+cu128. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling

## Paso 6 · Resultados y costo proyectado

In [8]:
import pandas as pd
ticks = [(int(a), float(b), float(c)) for a, b, c in
         re.findall(r"tick\s+(\d+)\s+kimg\s+([\d.]+).*?sec/kimg\s+([\d.]+)", log_txt)]
gpumem = [float(x) for x in re.findall(r"gpumem\s+([\d.]+)", log_txt)]
utiles = [s for t, k, s in ticks if t >= 1]
GAN = {
    "ticks_registrados": len(ticks),
    "kimg_alcanzado": ticks[-1][1] if ticks else 0.0,
    "sec_por_kimg": round(float(np.median(utiles)), 2) if utiles else None,
    "img_por_s": round(1000 / float(np.median(utiles)), 1) if utiles else None,
    "vram_pico_gb": max(gpumem) if gpumem else None,
    "extensiones_cuda": COMPILACION,
    "fallback_en_log": sorted(set(re.findall(r"Failed to build CUDA kernels for (\w+)", log_txt))),
    "errores": re.findall(r"(?m)^\w*Error.*$", log_txt)[:5],
}
filas = []
if GAN["sec_por_kimg"]:
    for kimg in [1000, 2000, 3000, 5000]:
        h = kimg * GAN["sec_por_kimg"] / 3600
        filas.append({"tecnica": "GAN", "escenario": f"{kimg} kimg", "horas_L4": round(h, 1),
                      "unidades": round(h * UNITS_PER_HOUR, 1)})
COSTO_GAN = pd.DataFrame(filas)

log03b = {"fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
          "versiones": {"python": sys.version.split()[0], "torch": torch.__version__,
                        "stylegan2_ada_commit": SG_COMMIT, "stylegan3_commit_gradfix": SG3_COMMIT},
          "parches": ["distutils.strtobool", "Sampler.__init__", "grid_sample_gradfix de StyleGAN3 sin pkg_resources", "custom_ops usa el módulo devuelto por cpp_extension.load"],
          "gan": GAN, "proyeccion_costos": filas, "units_per_hour": UNITS_PER_HOUR}
(LOGS / "03b_piloto_stylegan_v2.json").write_text(json.dumps(log03b, indent=2, ensure_ascii=False, default=str))
print(json.dumps(GAN, indent=2, ensure_ascii=False, default=str))
COSTO_GAN

{
  "ticks_registrados": 8,
  "kimg_alcanzado": 28.0,
  "sec_por_kimg": 26.03,
  "img_por_s": 38.4,
  "vram_pico_gb": 9.39,
  "extensiones_cuda": {
    "bias_act": {
      "compilado": true,
      "segundos": 38.0
    },
    "upfirdn2d": {
      "compilado": true,
      "segundos": 22.7
    }
  },
  "fallback_en_log": [],
  "errores": []
}


,tecnica,escenario,horas_L4,unidades
0,GAN,1000 kimg,7.2,13.0
1,GAN,2000 kimg,14.5,26.0
2,GAN,3000 kimg,21.7,39.0
3,GAN,5000 kimg,36.2,65.1


**Al terminar:** desconecta la GPU (*Entorno de ejecución → Desconectar y borrar*) y envía al asesor la salida del Paso 6 (y la del Paso 4 si alguna extensión no compiló).